# Conformación dataset

In [1]:
import glob
import xarray as xr
import geopandas as gpd
import googlemaps
import numpy as np
import pandas as pd
import os

In [2]:
import geopandas as gpd
import glob
import xarray as xr

# Ruta al archivo shape
shapefile_path = "/Users/riperez/Conda/anaconda3/doc/precipitation/shapes/MGN_MPIO_POLITICO.shp"

# Carpeta que contiene los archivos .nc
nc_folder_path = "/Users/riperez/Conda/anaconda3/doc/precipitation/CHIRPS-2.0/daily/"

# Cargar el archivo shape
gdf = gpd.read_file(shapefile_path)
print("Columnas en el shapefile:", gdf.columns)

# Verificar si el GeoDataFrame no está vacío
if gdf.empty:
    raise ValueError("El GeoDataFrame está vacío después de filtrar por 'BOYACÁ'. Verifica los nombres de los departamentos.")

# Conservar solo la geometría
gdf = gdf[['geometry']]

# Obtener coordenadas extremas (bounding box)
min_longitude, min_latitude, max_longitude, max_latitude = gdf.total_bounds

# Lista para guardar los datasets filtrados
filtered_datasets = []

# Lista de archivos NetCDF en la carpeta
nc_file_paths = glob.glob(nc_folder_path + "*.nc")

# Iterar sobre los archivos .nc
for nc_file_path in nc_file_paths:
    ds = xr.open_dataset(nc_file_path)
    
    # Filtrar por latitud y longitud
    ds_filtered = ds.sel(
        longitude=slice(min_longitude, max_longitude),
        latitude=slice(min_latitude, max_latitude)
    )
    
    filtered_datasets.append(ds_filtered)

# Combinar los datasets
combined_dataset = xr.concat(filtered_datasets, dim="time")

# Validar dimensiones
if combined_dataset.dims.get("latitude", 0) == 0 or combined_dataset.dims.get("longitude", 0) == 0:
    raise ValueError("El dataset combinado tiene dimensiones inválidas para latitud o longitud.")

# Guardar el dataset resultante
output_nc_path = "../data/data_filtered.nc"
combined_dataset.to_netcdf(output_nc_path)

print("✅ Dataset guardado exitosamente en:", output_nc_path)


Columnas en el shapefile: Index(['geometry'], dtype='object')


/var/folders/83/c6n8lktn4qx_fwp7ksllkkhn0dhtn2/T/ipykernel_52702/2106336813.py:47: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  if combined_dataset.dims.get("latitude", 0) == 0 or combined_dataset.dims.get("longitude", 0) == 0:
<frozen _collections_abc>:807: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.


✅ Dataset guardado exitosamente en: ../data/data_filtered.nc


In [3]:
data_filtered = xr.open_dataset("../data/data_filtered.nc")

In [4]:
# Crear un dataframe a partir del archivo .nc en chunks sin guardar archivos temporales
# Definir el tamaño del chunk
chunk_size = 100

# Iterar sobre las dimensiones del dataset en chunks
lat_chunks = data_filtered.latitude.values
lon_chunks = data_filtered.longitude.values

# Dividir las coordenadas en chunks
lat_chunks = [lat_chunks[i:i + chunk_size] for i in range(0, len(lat_chunks), chunk_size)]
lon_chunks = [lon_chunks[i:i + chunk_size] for i in range(0, len(lon_chunks), chunk_size)]

# DataFrame acumulativo
df_filtered = pd.DataFrame()

i = 0
for lat_chunk in lat_chunks:
    for lon_chunk in lon_chunks:
        # Seleccionar el subset del dataset
        chunk_ds = data_filtered.sel(latitude=lat_chunk, longitude=lon_chunk)
        
        # Convertir el chunk a un DataFrame y seleccionar las columnas necesarias
        chunk_df = chunk_ds.to_dataframe().reset_index()[['latitude', 'longitude']]
        
        # Agregar el chunk al DataFrame acumulativo
        df_filtered = pd.concat([df_filtered, chunk_df], ignore_index=True)

# Imprimir las columnas disponibles en df_filtered antes de trabajarlas
print("Columnas disponibles en df_filtered:", df_filtered.columns)

Columnas disponibles en df_filtered: Index(['latitude', 'longitude'], dtype='object')


In [ ]:
# Exportar el DataFrame a un archivo .nc
ds_boyaca = df_filtered.to_xarray()
ds_boyaca.to_netcdf('../data/data_boyaca_final.nc')

In [7]:
import matplotlib.pyplot as plt
import pandas as pd

# Cargar el dataset final
ds_boyaca = xr.open_dataset('../data/data_boyaca_final.nc')
df_boyaca = ds_boyaca.to_dataframe().reset_index()

# Convertir la columna de tiempo a tipo datetime
df_boyaca['time'] = pd.to_datetime(df_boyaca['time'])

# Agregar una columna para el mes
df_boyaca['month'] = df_boyaca['time'].dt.month

# Calcular estadísticas mensuales
monthly_stats = df_boyaca.groupby('month')['precipitation'].agg(['mean', 'max', 'min']).reset_index()

# Dibujar las estadísticas mensuales
plt.figure(figsize=(12, 6))
plt.plot(monthly_stats['month'], monthly_stats['mean'], label='Promedio', marker='o')
plt.plot(monthly_stats['month'], monthly_stats['max'], label='Máximo', marker='o')
plt.plot(monthly_stats['month'], monthly_stats['min'], label='Mínimo', marker='o')

# Configurar el gráfico
plt.title('Estadísticas Mensuales de Precipitación')
plt.xlabel('Mes')
plt.ylabel('Precipitación')
plt.xticks(range(1, 13))
plt.legend()
plt.grid(True)
plt.show()

: 